# 06 Encoding

## Train-Test Split (80/20)

This section:
- Loads the cleaned processed dataset
- Splits into train/test using `test_size=0.2` and `random_state=42`
- Sets aside the test set from this point onward
- Saves both datasets under `../data/processed/splits/`


In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
data_path = Path('../data/processed/ElectraHub_Data_cleaned.csv')
if not data_path.exists():
    raise FileNotFoundError(f'Processed dataset not found at: {data_path}')

df = pd.read_csv(data_path)
print(f'Loaded dataset: {data_path.name}')
print(f'Total rows: {len(df):,}')

Loaded dataset: ElectraHub_Data_cleaned.csv
Total rows: 3,000


In [3]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(f'Training rows: {len(train_df):,}')
print(f'Test rows: {len(test_df):,}')

Training rows: 2,400
Test rows: 600


In [4]:
splits_dir = Path('../data/processed/splits')
splits_dir.mkdir(parents=True, exist_ok=True)

train_path = splits_dir / 'ElectraHub_train.csv'
test_path = splits_dir / 'ElectraHub_test.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f'Saved training set to: {train_path}')
print(f'Saved test set to: {test_path}')
print('Test set has been set aside for later evaluation.')

Saved training set to: ../data/processed/splits/ElectraHub_train.csv
Saved test set to: ../data/processed/splits/ElectraHub_test.csv
Test set has been set aside for later evaluation.


## Proposed Encoding Plan (Review Before Implementation)

We will fit encoders on the **training set only** and apply the same fitted mapping to the test set.

### Categorical Columns and Recommended Encoding

| Column | Recommended Method | Why |
|---|---|---|
| `Region` | One-Hot Encoding (`drop='first'`, `handle_unknown='ignore'`) | Nominal categories with no natural order. One-hot avoids imposing fake numeric ordering. `drop='first'` reduces redundancy for linear models. |
| `Product_Category` | Binary encoding via One-Hot (`drop='first'`) | Two nominal levels (`Mobile`, `Tablet`), so one binary indicator is sufficient and interpretable. |
| `Campaign_Type` | One-Hot Encoding (`drop='first'`, `handle_unknown='ignore'`) | Nominal marketing channel categories; one-hot preserves category-specific effects for regression. |
| `Popularity` | **Ordinal Encoding** with explicit order: `Very Low < Low < Moderate < High < Very High` | This variable is inherently ordered, so ordinal encoding captures monotonic rank information more compactly than one-hot. |

### Recommended Implementation Notes

1. Fit all encoding transforms on `train_df` only.
2. Apply to both `train_df` and `test_df` using the fitted encoders.
3. Keep `Sales` untouched as target.
4. Save encoded outputs separately, e.g.:
   - `../data/processed/splits/ElectraHub_train_encoded.csv`
   - `../data/processed/splits/ElectraHub_test_encoded.csv`

### Caution Flags

- If `Popularity` does **not** behave monotonically with `Sales` after modeling checks, test a one-hot alternative for `Popularity` as a robustness comparison.
- Always use `handle_unknown='ignore'` for one-hot columns to avoid inference failures on unseen categories.


## Apply Encoding (Train Fit, Test Transform)

This section fits encoders on training data only and applies the same transformations to test data.


In [5]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [6]:
# Load train/test splits created earlier
train_path = Path('../data/processed/splits/ElectraHub_train.csv')
test_path = Path('../data/processed/splits/ElectraHub_test.csv')

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError('Train/test split files not found in ../data/processed/splits')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f'Train loaded: {train_df.shape[0]:,} rows x {train_df.shape[1]} cols')
print(f'Test loaded : {test_df.shape[0]:,} rows x {test_df.shape[1]} cols')

Train loaded: 2,400 rows x 16 cols
Test loaded : 600 rows x 16 cols


In [7]:
# Define target and categorical columns
TARGET_COL = 'Sales'
onehot_cols = ['Region', 'Product_Category', 'Campaign_Type']
ordinal_col = 'Popularity'

# Explicit ordinal mapping (fit choice fixed by business order)
popularity_order = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
popularity_map = {k: i for i, k in enumerate(popularity_order)}

# Separate target
y_train = train_df[TARGET_COL].copy()
y_test = test_df[TARGET_COL].copy()

X_train = train_df.drop(columns=[TARGET_COL]).copy()
X_test = test_df.drop(columns=[TARGET_COL]).copy()

# Ordinal encode Popularity
X_train[ordinal_col] = X_train[ordinal_col].astype(str).str.strip().map(popularity_map)
X_test[ordinal_col] = X_test[ordinal_col].astype(str).str.strip().map(popularity_map)

if X_train[ordinal_col].isna().any() or X_test[ordinal_col].isna().any():
    raise ValueError('Unmapped Popularity values found. Check category labels before encoding.')

# One-hot encode nominal columns (fit on train only)
ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

ohe_train = ohe.fit_transform(X_train[onehot_cols])
ohe_test = ohe.transform(X_test[onehot_cols])

ohe_cols = ohe.get_feature_names_out(onehot_cols)

ohe_train_df = pd.DataFrame(ohe_train, columns=ohe_cols, index=X_train.index)
ohe_test_df = pd.DataFrame(ohe_test, columns=ohe_cols, index=X_test.index)

# Remaining numeric + ordinal columns
base_cols = [c for c in X_train.columns if c not in onehot_cols]

X_train_encoded = pd.concat([X_train[base_cols], ohe_train_df], axis=1)
X_test_encoded = pd.concat([X_test[base_cols], ohe_test_df], axis=1)

# Reattach target at the end
train_encoded = pd.concat([X_train_encoded, y_train], axis=1)
test_encoded = pd.concat([X_test_encoded, y_test], axis=1)

# Ensure same columns/order
test_encoded = test_encoded[train_encoded.columns]

print(f'Encoded train shape: {train_encoded.shape[0]:,} x {train_encoded.shape[1]}')
print(f'Encoded test shape : {test_encoded.shape[0]:,} x {test_encoded.shape[1]}')

Encoded train shape: 2,400 x 20
Encoded test shape : 600 x 20


In [8]:
# Save encoded datasets to ../data/processed
out_dir = Path('../data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

train_encoded_path = out_dir / 'ElectraHub_train_encoded.csv'
test_encoded_path = out_dir / 'ElectraHub_test_encoded.csv'

train_encoded.to_csv(train_encoded_path, index=False)
test_encoded.to_csv(test_encoded_path, index=False)

print(f'Saved encoded train: {train_encoded_path}')
print(f'Saved encoded test : {test_encoded_path}')

print('\nFinal encoded columns:')
print(list(train_encoded.columns))

Saved encoded train: ../data/processed/ElectraHub_train_encoded.csv
Saved encoded test : ../data/processed/ElectraHub_test_encoded.csv

Final encoded columns:
['Product_Age_Months', 'Product_Price', 'Competitor_Price_Index', 'Advertising_Expenditure', 'Discount_Percentage', 'Campaign_Engagement_Score', 'Inventory_Level', 'Num_Reviews', 'Avg_Customer_Rating', 'Return_Rate', 'Length_Product_Description', 'Popularity', 'Region_North', 'Region_South', 'Region_West', 'Product_Category_Tablet', 'Campaign_Type_Influencer', 'Campaign_Type_Search Ad', 'Campaign_Type_Social Media', 'Sales']
